# Linear Regression

Linear regression is an approach we use to model the relationship between a scalar dependent variable *`y`* and one or more explanatory variables (aka independent variables) denoted by *`X`*. In machine learning parlance, these explanatory variables or predictors are called **features** and target or the dependent variable **labels**. Regression modesl are models which predict a continuous outcome.

## Univariate Regression
The simplest possible relationship between two variables is a *`univariate regression`* when there is only one feature to be explained.<br><br>

$$y_i \approx w_0 + w_1x_i$$

If we can measure some $(x_i, y_i)$ pairs, we could calculate our _model parameters_ $w_0$ and $w_1$. Then we could predict $y$ in the future based on $x$, or even try to influence $y$ in the future by controlling $x$. This is achieved by minimizing the total squared error, <br><br>  

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2$$

## Multivariate Regression
If there multiple features that are to be mapped to a target variable, then it is a *`multivariate regression`* problem.<br><br>  

$$ y_i \approx w_0 + w_1x_{1} + w_2x_{2} + w_3x_{3} + \cdots + w_nx_{n} $$

or more generally

$$ y_i \approx  \sum_jw_jX_{ij} $$

where $i$ labels different **observations** and $j$ labels different **features**.

---

_**Note:** Some third-party Python libraries (e.g., yfinance, quantmod) depend on external data sources that may be restricted or inaccessible in certain regions, including China. Users in these regions are advised to consult their local instructor or substitute with locally supported alternatives._

---


**Install Packages**

In [ ]:
!python -m ensurepip --upgrade
!python -m pip install --upgrade pip
!pip install numpy wandb quantmod pandas


Looking in links: c:\Users\admin\AppData\Local\Temp\tmp02auxp0x
Processing c:\users\admin\appdata\local\temp\tmp02auxp0x\setuptools-79.0.1-py3-none-any.whl
Processing c:\users\admin\appdata\local\temp\tmp02auxp0x\pip-24.0-py3-none-any.whl
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   --------- ------------------------------ 0.5/1.8 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------  1.8/1.8 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 16.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------------------------- -------- 10.0/12.6 MB 47.8 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 39.6 MB/s  0:00:00
   ---------------------------------------- 0.0/24.1 MB ? eta -:--:--
   ------------ --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hermes-agent 0.19.1 requires urllib3<3,>=2.7.0, but you have urllib3 1.26.15 which is incompatible.


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

**Import Libraries**

In [3]:
# Data Manipulation
import numpy as np
import pandas as pd

# Quantmod
import quantmod.charts
from quantmod.markets import getData
from quantmod.datasets import fetch_historical_data
from quantmod.timeseries import OpCl, HiLo, Gap, dailyReturn, lag, lead
from quantmod.indicators import SMA, BBands


# scikit-learn
import sklearn

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline

# Regressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

# Metrics
from sklearn.metrics import mean_squared_error

# Import Libraries
# from google.colab import userdata

## Section 1: Experiment Tracking

Experiment tracking is a foundational practice in data science, machine learning, and quantitative research. As experiments grow in complexity - spanning multiple datasets, models, hyperparameters, and evaluation metrics - it becomes increasingly difficult to reproduce results or understand what actually worked. This is where experiment trackers come in.

> _An experiment tracker is a tool or framework that systematically records every aspect of an experiment: code versions, input data, parameter configurations, outputs, metrics, and environment details. Instead of relying on ad-hoc notes or scattered logs, it enables a structured, searchable history of our work, improving reproducibility while accelerating iteration and collaboration._

Popular platforms such as MLflow and Weights & Biases provide both local and cloud-based solutions, supporting everything from lightweight logging to full-scale experiment management.

In our labs, we use Weights & Biases (**W&B**), which integrates seamlessly with Python workflows, enabling real-time logging, run comparison, and artifact management. It brings discipline and visibility to experimentation, making iteration faster and results more reliable.

Create an account on [Weights & Biases](https://wandb.ai/), set up a project, and generate your API key to get started. Save it in Colab Secrets to enable seamless authentication.

In [8]:
import wandb

# Replace the string with your actual WandB API key from your account settings
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\admin\_netrc
wandb: Currently logged in as: abhaydduk (abhaydduk-deutsche-bank) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Section 2: The workflow

We'll employ the following linear regressor from `scikit-learn` for price prediction.

* Linear Regression
* Lasso Regression
* Ridge Regression
* Elastic Net Regression

| Steps        | Workflow                  | Remarks                                                         |
|:-------------|:--------------------------|:----------------------------------------------------------------|
|Step 1        | Ideation                  | Define objective, success metrics     |
|Step 2        | Data Collection           | Gather and integrate data
|Step 3        | Exploratory Data Analysis (Initial) | Broad exploration: stats, distributions, correlations, missing data |
|Step 4        | Data Cleaning           | Handle missing values, outliers, duplicates.            |
|Step 5        | Feature Engineering & Transformation            | Feature creation, scaling, encoding, selection                         |
|        | Subset Validation EDA            | Re-examine chosen features: check distributions, multicollinearity, relationships                      |               
|Step 6        | Modeling                  | Select algorithm(s), train models, tune hyperparameters                           |
|Step 7        | Evaluation                   | Validate using metrics and backtesting       |

### (1) Load Data

In [9]:
# Load sample data from quandmod
df = (
    fetch_historical_data("NIFTY")
    .assign(date=lambda df_: pd.to_datetime(df_["date"]))
    .set_index('date')
    .copy()
)

df

,open,high,low,close,volume
date,,,,,
2012-01-02,4640.20,4645.95,4588.05,4636.75,108461000
2012-01-03,4675.80,4773.10,4675.80,4765.30,146621008
2012-01-04,4774.95,4782.85,4728.85,4749.65,165939008
2012-01-05,4749.00,4779.80,4730.15,4749.95,177863008
2012-01-06,4724.15,4794.90,4686.85,4754.10,176057008
...,...,...,...,...,...
2022-12-26,17830.40,18084.10,17774.25,18014.60,176652336
2022-12-27,18089.80,18149.25,17967.45,18132.30,214257088
2022-12-28,18084.75,18173.10,18068.35,18122.50,193873168


In [10]:
df.index

DatetimeIndex(['2012-01-02', '2012-01-03', '2012-01-04', '2012-01-05',
               '2012-01-06', '2012-01-09', '2012-01-10', '2012-01-11',
               '2012-01-12', '2012-01-13',
               ...
               '2022-12-19', '2022-12-20', '2022-12-21', '2022-12-22',
               '2022-12-23', '2022-12-26', '2022-12-27', '2022-12-28',
               '2022-12-29', '2022-12-30'],
              dtype='datetime64[us]', name='date', length=2719, freq=None)

### (2) EDA of Original dataset

In [11]:
# Descriptive Statistics
df.describe().T

,count,mean,std,min,25%,50%,75%,max
open,2719.0,1.012490e+04,3.709607e+03,4640.20,7.637000e+03,9.376950e+03,1.170085e+04,1.887195e+04
high,2719.0,1.017529e+04,3.725527e+03,4645.95,7.681000e+03,9.431900e+03,1.175512e+04,1.888760e+04
low,2719.0,1.005747e+04,3.685305e+03,4588.05,7.581650e+03,9.301350e+03,1.163133e+04,1.877820e+04
close,2719.0,1.011757e+04,3.706945e+03,4636.75,7.623800e+03,9.351850e+03,1.168400e+04,1.881250e+04
volume,2719.0,2.892221e+08,1.952768e+08,2826000.00,1.591780e+08,2.194961e+08,3.409088e+08,1.810971e+09


### (3) Cleaning & Imputation

Data is already cleaned. No further processing or imputation required.

In [12]:
# Check for missing values
df.isnull().sum()

open      0
high      0
low       0
close     0
volume    0
dtype: int64

### (4) Feature Engineering
Features are also known as an independent variable which are used to determine the value of the target variable. We will now create features and target (label) from the raw dataset.

In [13]:
# Define Create Features
def create_features(frame):
    df = frame.copy()
    multiplier = 2

    # Features - X
    df['OC'] = OpCl(df)
    df['HC'] = HiLo(df)
    df['GAP'] = Gap(df)
    df['RET'] = dailyReturn(df.close)

    for i in [7, 14, 28]:
        df['PCHG' + str(i)] = df['close'].pct_change(i)
        df['VCHG' + str(i)] = df['volume'].pct_change(i)
        df['RET' + str(i)] = df['RET'].rolling(i).sum()
        df['MA' + str(i)] = SMA(df['close'], i)
        # df['VMA' + str(i)] = SMA(df['volume'], i)
        df['OC'+ str(i)] = df['OC'].rolling(i).mean()
        df['HC'+ str(i)] = df['HC'].rolling(i).mean()
        df['GAP'+ str(i)] = df['GAP'].rolling(i).mean()
        df['STD'+ str(i)] = df['RET'].rolling(i).std()
        df['LB'+str(i)] = BBands(df['close'], i, multiplier)[0]
        df['UB'+str(i)] = BBands(df['close'], i, multiplier)[2]

    # Define the label - y
    df['Label'] = lead(df['close'])

    # Drop unnecessary columns - important
    df.drop(['open', 'high', 'low', 'close', 'volume'], axis=1, inplace=True)
    df.dropna(inplace=True)

    return df

#### (a) Feature Specification

In [14]:
# Features
df1 = create_features(df)
display(df1.shape)

# Verify the output
df1.head(2)

(2690, 35)

,OC,HC,GAP,RET,PCHG7,VCHG7,RET7,MA7,OC7,HC7,...,VCHG28,RET28,MA28,OC28,HC28,GAP28,STD28,LB28,UB28,Label
date,,,,,,,,,,,,,,,,,,,,,
2012-02-10,18.2,86.70,-0.002319,-0.005698,0.027866,-0.040643,0.027485,5350.664286,-3.914286,77.085714,...,1.077724,0.148972,5055.251786,-14.250000,68.944643,0.002448,0.010362,4600.976081,5509.527490,5390.20
2012-02-13,-8.1,69.65,0.000093,0.001597,0.022828,-0.462772,0.022571,5367.850000,-5.385714,77.864286,...,0.238308,0.123222,5077.569643,-11.342857,67.957143,0.002151,0.009437,4620.988930,5534.150356,5416.05


#### (b) Target or Label Definition
Label or the target variable is also known as the dependent variable. Here, the target variable is the closing price of the index the next trading day.

In [15]:
# Label
y = np.array(df1['Label'])
y

array([ 5390.2 ,  5416.05,  5531.95, ..., 18122.5 , 18191.  , 18105.3 ],
      shape=(2690,))

#### (c) Feature Selection
Reduce the number of input variables that are believed to be most useful to a model. We use correlation measures which is one the filter methods to address multicollinearity among features.

In [16]:
# Drop label from dataframe
df1.drop('Label', axis=1, inplace=True)

# Plot correlation matrix
df1.corr().iplot(
    kind="heatmap",
    title="Features Set Correlations",
    width=1000,
    height=800
    )

In [17]:
# Remove highly correlated features - greater than 0.90 in this case
threshold = 0.9
corr_matrix = df1.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns to drop
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

# Number of features dropped
print(f"\n No. of features dropped: {len(to_drop)} \n List of features dropped: {to_drop}")


 No. of features dropped: 13 
 List of features dropped: ['RET7', 'LB7', 'UB7', 'RET14', 'MA14', 'HC14', 'LB14', 'UB14', 'RET28', 'MA28', 'HC28', 'LB28', 'UB28']


**Selected Features**

In [18]:
# Drop the highly correlated features
X = df1.drop(to_drop, axis=1)

# record feature names
feature_names = X.columns

In [19]:
# Display the new features set
X.head()

,OC,HC,GAP,RET,PCHG7,VCHG7,MA7,OC7,HC7,GAP7,...,PCHG14,VCHG14,OC14,GAP14,STD14,PCHG28,VCHG28,OC28,GAP28,STD28
date,,,,,,,,,,,,,,,,,,,,,
2012-02-10,18.20,86.70,-0.002319,-0.005698,0.027866,-0.040643,5350.664286,-3.914286,77.085714,0.003198,...,0.065959,0.019471,-10.064286,0.002632,0.010683,0.160641,1.077724,-14.250000,0.002448,0.010362
2012-02-13,-8.10,69.65,0.000093,0.001597,0.022828,-0.462772,5367.850000,-5.385714,77.864286,0.002218,...,0.068160,0.147833,-9.150000,0.002967,0.010622,0.131136,0.238308,-11.342857,0.002151,0.009437
2012-02-14,-35.25,50.10,-0.001744,0.004784,0.016936,-0.073372,5380.735714,-3.314286,73.692857,0.001801,...,0.056306,-0.002955,-7.200000,0.002580,0.010121,0.140305,0.213759,-13.505357,0.002016,0.009315
2012-02-15,-71.35,81.50,0.008226,0.021174,0.031763,0.461624,5405.064286,-16.050000,76.364286,0.001538,...,0.072437,0.434309,-11.810714,0.002831,0.011125,0.164633,0.767461,-16.019643,0.002315,0.009770
2012-02-16,-8.20,47.65,-0.003290,-0.001809,0.035013,0.490439,5431.750000,-28.335714,70.257143,-0.000298,...,0.060955,0.250724,-13.257143,0.001787,0.011202,0.161513,0.533935,-15.242857,0.002391,0.009830


### (5) EDA of Subset Selection

> The choice of scaling techniques to be used should come from the Exploratory Data Analysis of features set.

In [20]:
# decide which scaling to use
X.describe()

,OC,HC,GAP,RET,PCHG7,VCHG7,MA7,OC7,HC7,GAP7,...,PCHG14,VCHG14,OC14,GAP14,STD14,PCHG28,VCHG28,OC28,GAP28,STD28
count,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,...,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000,2690.000000
mean,7.502584,118.316803,0.001190,0.000451,0.003593,0.101616,10153.370467,7.556208,118.170459,0.001194,...,0.007386,0.165838,7.484822,0.001200,0.009347,0.015286,0.122934,7.413427,0.001208,0.009542
std,91.401173,87.811119,0.006300,0.010769,0.028319,1.947073,3678.835498,31.317749,68.962550,0.002593,...,0.040495,4.865908,21.570305,0.001982,0.005297,0.056592,2.228693,15.790333,0.001375,0.004996
min,-847.600000,21.850000,-0.091448,-0.139038,-0.229370,-0.992479,4877.642857,-114.185714,35.071429,-0.028701,...,-0.316409,-0.990374,-68.607143,-0.020488,0.002493,-0.367473,-0.991950,-37.044643,-0.009985,0.003677
25%,-38.887500,64.100000,-0.000997,-0.004635,-0.011853,-0.181148,7688.773214,-10.978571,72.550000,0.000226,...,-0.013799,-0.199597,-5.519643,0.000437,0.006466,-0.016429,-0.209351,-2.622768,0.000516,0.006760
50%,4.775000,93.725000,0.001507,0.000687,0.005298,0.002136,9444.553571,5.446429,98.060714,0.001345,...,0.009224,-0.005628,6.037500,0.001335,0.008407,0.018710,-0.011725,5.567857,0.001323,0.008685
75%,47.287500,143.650000,0.003930,0.006208,0.019829,0.212967,11689.028571,23.021429,143.400000,0.002427,...,0.031227,0.229800,17.574107,0.002170,0.010485,0.051332,0.255204,14.558036,0.001958,0.010767
max,619.650000,1604.250000,0.044843,0.084003,0.146336,98.264679,18684.507143,156.728571,753.342857,0.015550,...,0.187885,251.176577,111.767857,0.013068,0.061454,0.208771,112.871909,78.205357,0.007392,0.049699


In [21]:
X.iplot(
    kind="box",
    showlegend=True,
    title="Box Plot Analysis",
    width=1500,
    height=800
    )

### (6) Linear Regression
Linear regression is an approach we use to model the relationship between a scalar dependent variable *`y`* and one or more explanatory variables (aka independent variables) denoted by *`X`*.

**Train - Test Data**

In [22]:
# Splitting the datasets into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Output the train and test data size
print(f"Train and Test Size {len(X_train)}, {len(X_test)}")

Train and Test Size 2152, 538


In [23]:
# Use pipeline to define the linear regression model
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

lr

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None


In [24]:
# Fit/Train model
lr.fit(X_train, y_train)

# Predict labels
y_pred = lr.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'MSE: {mse:0.4}')
print(f'RMSE: {rmse:0.4}')
print(f'R^2 Train: {lr.score(X_train, y_train):0.4}')
print(f'R^2 Test: {lr.score(X_test, y_test):0.4}')

MSE: 4.078e+04
RMSE: 201.9
R^2 Train: 0.9974
R^2 Test: 0.9826


In [25]:
# Get the coefficients or weights of the model
lr_coef = lr['regressor'].coef_
lr_coef

array([-2.32326560e+01, -2.56710670e+01,  1.34698045e+01,  1.70443940e+01,
       -2.08026396e+01, -7.94342871e-01,  2.11795230e+03, -7.45375573e+01,
        9.10957686e+00,  7.91331641e+01,  2.86879354e+01, -3.18452966e+01,
        2.37181354e+00, -3.27177486e+01,  2.36988901e+01, -8.87843516e+00,
        1.09655352e+01, -2.32119174e+00,  7.76229473e+00, -1.00391173e+01,
       -4.56939148e+00])

In [26]:
# Get the intercept of the regression model
lr['regressor'].intercept_

np.float64(8625.386710037175)

**Regularized Regression**

If the coeficients are too large in linear regression, it can lead to overfitting the model. We do regularization to over come such issues that penalises large coefficients. Lasso and Ridge regression are penalty regression that prevent over-fitting from the simple linear regression.

#### (a) LASSO
The Least Absolute Shrinkage and Selection Operator (LASSO) is a variation of linear regression. In Lasso, the loss function is minimized by limiting the sum of absolute values of the model coefficients where the L1 penalty term is added to the Residual Sum of Squares (RSS). The L1 penalty term not only shrinks the coefficents, but shrinks some of them to zero and that is very useful for feature selection. <br><br>

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \sum_{j=1}^{p} {\mid w_j\mid}$$

where,  $\lambda$ is the regularization penalty.

In [27]:
# Use pipeline to define the Lasso model
lasso = Pipeline([
    ('scaler', MinMaxScaler()),
    ('regressor', Lasso(alpha=0.1))
])

# Fit lasso model
lasso.fit(X_train, y_train)

# Prediction
lasso_pred = lasso.predict(X_test)

print(f'R^2 Train: {lasso.score(X_train, y_train):0.4}')
print(f'R^2 Test: {lasso.score(X_test, y_test):0.4}')

R^2 Train: 0.9974
R^2 Test: 0.9827


In [28]:
# Get the coefficients or weights of the model
lasso_coef = lasso['regressor'].coef_
lasso_coef

array([-3.16065519e+02, -4.47303031e+02,  1.94699374e+02,  4.82678721e+02,
        0.00000000e+00, -0.00000000e+00,  7.47371223e+03, -5.87325894e+02,
        0.00000000e+00,  1.06170521e+03,  2.33531891e+02, -0.00000000e+00,
        0.00000000e+00, -7.23523286e+01,  0.00000000e+00, -0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00, -7.51567323e+00, -0.00000000e+00,
       -2.71350779e+00])

In [29]:
# Log Lasso coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="lasso-coef-path",
    job_type="coef-path",
    config={
        "model": "Lasso",
        "scaler": "StandardScaler",
        "alpha_min": 1e-2,
        "alpha_max": 1e2,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(2, -2, 200)

for alpha in alpha_range:
    las = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Lasso(alpha=alpha, max_iter=10000))
    ])
    las.fit(X_train, y_train)
    y_pred = las.predict(X_test)

    w = las.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": las.score(X_train, y_train),
           "test_r2": las.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
           "n_nonzero": int(np.sum(w != 0))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

# Log the full coefficient path as a table
wandb.log({"lasso_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
run.finish()

alpha,█▇▆▆▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/GAP,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▅▅▆▇▇███████▇▇▇▇▇▇▇▇▇▇
coef/GAP14,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▄▅▅▅▆▆▆▇▇▇▇▇███
coef/GAP28,█████████████████████████████▇▇▇▆▆▆▅▃▃▂▁
coef/GAP7,▁▁▁▁▁▂▂▂▂▂▂▂▃▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
coef/HC,██████████▇▆▆▅▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/HC7,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▆▆▇██████████
coef/MA7,▁▂▃▃▄▅▅▆▆▇▇█████████████████████████████
coef/OC,████████████▅▅▄▃▂▂▂▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂
coef/OC14,███████▇▇▇▇▇▆▆▆▆▆▆▆▆▆▆▅▅▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
+17,...


#### (b) Ridge
In ridge regression, the cost function is altered by adding a L2 penality equivalent to square of the magnitude of the coefficients. The Ridge regression shrinks the coefficients and helps to reduce the multi-collinearity. As seen from the above graph, when $\lambda \xrightarrow {}0$, the cost function becomes similar to the linear regression cost function RSS.

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \sum_{j=1}^{p} w_j^2$$

In [30]:
# Use pipeline to define the ridge model
rid = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1))
])

# fit ridge model
rid.fit(X, y)

# predict
ridge_pred = rid.predict(X_test)

print(f'R^2 Train: {rid.score(X_train, y_train):0.4}')
print(f'R^2 Test: {rid.score(X_test, y_test):0.4}')

R^2 Train: 0.9973
R^2 Test: 0.9843


In [31]:
# Get the coefficients or weights of the model
ridge_coef = rid['regressor'].coef_
ridge_coef

array([-3.61587748e+01, -2.52736845e+01,  2.52359607e+01,  8.06820983e+00,
       -2.78070064e+01, -3.72672156e-01,  3.67702775e+03, -9.58803762e+01,
        3.59123841e+01,  9.54129079e+01,  2.15571125e+01,  2.50620183e+01,
        1.87771703e+00,  1.65418608e+01, -1.52688235e+01, -1.30801693e+01,
       -8.76099046e+00, -2.21816945e+00, -1.49262469e+01,  5.24616500e+00,
       -1.12456216e+01])

**Coefficient Analysis**

In [32]:
coef = []
alpha_range = np.logspace(6, -2, 200)

for alpha in alpha_range:
    # Define pipeline
    rid = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=alpha, max_iter=10000))
    ])

    # Fit pipeline on training data
    rid.fit(X_train, y_train)

    # Predict on test set if needed (optional)
    y_pred = rid.predict(X_test)

    # Store coefficients
    coef.append(rid.named_steps['regressor'].coef_)

# Create DataFrame: rows = alphas, columns = features
coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'


In [33]:
# plot coefficients
# coef_df.iplot(kind='overlay', showlegend=True)
fig = coef_df.drop('MA7', axis=1).iplot(
    kind='overlay',
    title='Coefficients as a function of the regularization',
    xaxis_title='λ (log scale)',
    yaxis_title='w',
    showlegend=True
    )

# plot figure
fig.show()

In [34]:
# Log Ridge coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="ridge-coef-path",
    job_type="coef-path",
    config={
        "model": "Ridge",
        "scaler": "StandardScaler",
        "alpha_min": 1e-2,
        "alpha_max": 1e6,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(6, -2, 200)

for alpha in alpha_range:
    rid = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=alpha, max_iter=10000))
    ])
    rid.fit(X_train, y_train)
    y_pred = rid.predict(X_test)

    w = rid.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": rid.score(X_train, y_train),
           "test_r2": rid.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

wandb.log({"ridge_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
wandb.log({"Coeff": fig})

run.finish()

alpha,█▄▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/GAP,▁▁▁▁▁▃▃▃▄▅▆▆▇██▇▇▆▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
coef/GAP14,▁▁▁▂▂▂▂▂▂▂▆▇████▇▆▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
coef/GAP28,▂▂▂▂▂▂▃▃▄███▇▅▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/GAP7,▁▁▁▁▁▂▂▂▄▄▄▄▆▆▆▆▆▇▇▇▇▇▇▇▇███████████████
coef/HC,▂▂▂▃▃▃▃▃▃▄▆▆▇██▇▇▆▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/HC7,▁▁▁▁▁▂▂▂▂▃▄▄▆▇▇████▇▆▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/MA7,▁▁▁▁▁▁▁▁▂▂▃▄▄▅▅▅▆▆▆▇▇███████████████████
coef/OC,▇▇▇▇▇▇▇▇██▇▇▇▇▆▅▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/OC14,▄▄▄▄▄▄▄▄▅▆▆▇████▇▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+16,...


#### (c) ElasticNet Regression
ElasticNet combines the properties of both Lasso and Ridge regression. It penalizes the model using both the L1 and L2 norm. <br><br>

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \left(\frac{1-\alpha}{2} \sum_{j=1}^{p} w_j^2 + \alpha \sum_{j=1}^{p} {\mid w_j\mid} \right)$$

where $0\leq \alpha \leq1$. Elastic net is same as lasso when $\alpha=1$ and ridge when $\alpha=0$. For other values of $\alpha$, the penalty term interpolate between L1 and L2 norm of the coefficient.

In [35]:
# Use pipeline to define the elastic net model
elastic = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', ElasticNet(alpha=0.1, l1_ratio=0.3))
])

# fit elasticnet model
elastic.fit(X_train, y_train)

# predict
elasticnet_pred = elastic.predict(X_test)

print(f'R^2 Train: {elastic.score(X_train, y_train):0.4}')
print(f'R^2 Test: {elastic.score(X_test, y_test):0.4}')

R^2 Train: 0.991
R^2 Test: 0.8263


In [36]:
# Get the coefficients or weights of the model
elasticnet_coef = elastic['regressor'].coef_
elasticnet_coef

array([-1.01936963e+01,  3.54209309e+00,  9.16418783e+00,  2.31192822e+01,
        5.73430237e+01,  6.75623041e-01,  1.86560269e+03, -2.81242750e+01,
        2.08994115e+02,  6.29600002e+01, -6.35944767e+01,  2.57091250e+01,
        4.54170686e+00,  2.44179817e+00,  1.77447362e+01, -7.01888389e+01,
        2.60389481e+01,  1.51624801e+00,  5.54155628e+01, -3.53132663e+00,
       -5.75689916e+01])

In [37]:
# Log ElasticNet coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="elasticnet-coef-path",
    job_type="coef-path",
    config={
        "model": "ElasticNet",
        "scaler": "StandardScaler",
        "l1_ratio": 0.5,
        "alpha_min": 1e-2,
        "alpha_max": 1e2,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(2, -2, 200)

for alpha in alpha_range:
    en = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', ElasticNet(alpha=alpha, max_iter=10000))
    ])
    en.fit(X_train, y_train)
    y_pred = en.predict(X_test)

    w = en.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": en.score(X_train, y_train),
           "test_r2": en.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
           "n_nonzero": int(np.sum(w != 0))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

wandb.log({"elasticnet_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
run.finish()

alpha,█▇▇▆▆▅▄▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
coef/GAP,▁▁▂▂▂▃▃▄▄▄▅▅▆▆▇█████▇▇▇▇▆▆▅▅▅▅▅▆▆▇▇▇▇▇▇▇
coef/GAP14,▁▁▂▂▂▃▄▄▄▅▆▆▇▇▇▇████▇▆▅▅▅▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂
coef/GAP28,▂▃▃▃▃▄▄▄▅▅▅▆▇▇▇████▇▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁
coef/GAP7,▁▁▁▁▁▂▂▂▃▃▄▄▅▅▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████
coef/HC,▃▃▃▃▃▃▄▄▄▄▅▆▆▇▇▇████▆▅▄▄▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
coef/HC7,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▇█████▇▆▆▆▅▅▄▄▄▃▃▂▂▂▂▁
coef/MA7,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▆▆▆▆▇▇▇▇▇██████████
coef/OC,▇▇▇▇▇▇▇▇▇▇████████▇▇▇▇▇▇▇▆▆▆▆▆▄▄▃▃▃▂▂▂▁▁
coef/OC14,▄▅▅▅▅▆▆▇▇▇▇██████▇▇▇▆▆▆▆▅▅▄▄▄▄▂▂▂▂▂▁▁▁▁▁
+17,...


### (7) Evaluation: Model Comparision

In [38]:
# compare model coefficients
# weights[weights['Lasso']!=0].index

weights = pd.DataFrame({
    'LR': lr_coef,
    'Lasso': lasso_coef,
    'Ridge': ridge_coef,
    'ElasticNet': elasticnet_coef
}, index = feature_names)

weights

,LR,Lasso,Ridge,ElasticNet
OC,-23.232656,-316.065519,-36.158775,-10.193696
HC,-25.671067,-447.303031,-25.273685,3.542093
GAP,13.469805,194.699374,25.235961,9.164188
RET,17.044394,482.678721,8.068210,23.119282
PCHG7,-20.802640,0.000000,-27.807006,57.343024
VCHG7,-0.794343,-0.000000,-0.372672,0.675623
MA7,2117.952295,7473.712225,3677.027751,1865.602691
OC7,-74.537557,-587.325894,-95.880376,-28.124275
HC7,9.109577,0.000000,35.912384,208.994115
GAP7,79.133164,1061.705213,95.412908,62.960000


## Section 3: Also Read

📌 [Quantmod](https://kannansingaravelu.com/quantmod/)

📌 [Scikit-learn](https://scikit-learn.org/stable/index.html)

📌 [Weights & Biases](https://docs.wandb.ai/)  

---
[Kannan Singaravelu](https://www.linkedin.com/in/kannansi)